# Executive Analytical Dashboard & Synthesis

**Role:** Cross-domain synthesis — this notebook aggregates the most decision-relevant findings from all five analytical themes into a single executive-level view.

**Business question:** What is the current state of the business across revenue, retention, operations, satisfaction, and payments — and where are the highest-priority risks?

**Audience:** Executive leadership, business stakeholders, and portfolio reviewers requiring a single starting point for the analysis.


## Data Sources

This notebook re-executes all five analytical query sets (Q1 from each) and aggregates the results into a unified KPI table. No new SQL queries are introduced.

| Theme | SQL File | Key Metrics |
|---|---|---|
| Revenue & AOV | `01_revenue_and_aov_behavior.sql` | Total GMV, total orders, avg AOV |
| Cohorts & LTV | `02_cohorts_and_retention.sql` | Avg month-1 retention, peak LTV |
| Delivery SLA | `03_delivery_sla_performance.sql` | Delay rate, ETA overshoot |
| Review Scores | `04_review_score_drivers.sql` | Score gap (on-time vs. delayed) |
| Payment Types | `05_payment_type_behavior.sql` | Credit card GMV share, cancel rate |

**Grain:** One row per business theme in the risk matrix. Scalar KPIs extracted from the first row of each summary query result.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

_REPO_ROOT = Path().resolve().parents[1]
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from analysis.utils.db import get_connection
from analysis.utils.sql_loader import get_sql_path, load_queries
from analysis.utils.plotting import apply_style, save_figure

apply_style()

# =============================================================================
# Notebook 10 — Executive Summary Dashboard
# Setup: re-execute all 5 analytical query sets; build cross-notebook KPI table
# =============================================================================

# ---------------------------------------------------------------------------
# Load all 5 analysis query sets
# ---------------------------------------------------------------------------
sql_files = {
    "revenue":  "sql/analysis/01_revenue_and_aov_behavior.sql",
    "cohorts":  "sql/analysis/02_cohorts_and_retention.sql",
    "sla":      "sql/analysis/03_delivery_sla_performance.sql",
    "reviews":  "sql/analysis/04_review_score_drivers.sql",
    "payments": "sql/analysis/05_payment_type_behavior.sql",
}

with get_connection() as conn:
    # Revenue & AOV
    q_rev = load_queries(get_sql_path(sql_files["revenue"]))
    df_monthly = pd.read_sql(q_rev[0], conn)
    df_cats    = pd.read_sql(q_rev[1], conn)

    # Cohorts
    q_coh = load_queries(get_sql_path(sql_files["cohorts"]))
    df_retention = pd.read_sql(q_coh[0], conn)
    df_ltv_exec  = pd.read_sql(q_coh[1], conn)

    # SLA
    q_sla = load_queries(get_sql_path(sql_files["sla"]))
    df_sla_exec  = pd.read_sql(q_sla[0], conn)
    df_delay_geo = pd.read_sql(q_sla[1], conn)

    # Reviews
    q_rev4 = load_queries(get_sql_path(sql_files["reviews"]))
    df_scores      = pd.read_sql(q_rev4[0], conn)
    df_del_scores  = pd.read_sql(q_rev4[1], conn)
    df_worst_cats  = pd.read_sql(q_rev4[2], conn)

    # Payments
    q_pay = load_queries(get_sql_path(sql_files["payments"]))
    df_payments = pd.read_sql(q_pay[0], conn)
    df_cancels  = pd.read_sql(q_pay[1], conn)

df_monthly["revenue_month"] = pd.to_datetime(df_monthly["revenue_month"])
df_retention["cohort_month"] = pd.to_datetime(df_retention["cohort_month"])

# ---------------------------------------------------------------------------
# Extract scalar KPIs for the executive scorecard
# ---------------------------------------------------------------------------

# Revenue
total_gmv_exec    = df_monthly["total_revenue"].sum()
total_orders_exec = df_monthly["total_orders"].sum()
avg_aov           = df_monthly["average_order_value"].mean()

# Retention
m1_ret_df   = df_retention[df_retention["months_since_first_purchase"] == 1]
avg_m1_ret  = m1_ret_df["retention_rate_pct"].mean() if len(m1_ret_df) > 0 else None
peak_ltv    = df_ltv_exec["avg_cumulative_ltv"].max()

# SLA
delayed_rate_exec = float(df_sla_exec["delayed_rate_pct"].iloc[0])
avg_overshot_days = (
    float(df_sla_exec["avg_actual_delivery_days"].iloc[0])
    - float(df_sla_exec["avg_estimated_delivery_days"].iloc[0])
)

# Reviews
score_on_time = None
score_delayed = None
del_idx = df_del_scores.set_index("delivery_status")
if "On-Time" in del_idx.index:
    score_on_time = float(del_idx.loc["On-Time", "avg_review_score"])
if "Delayed" in del_idx.index:
    score_delayed = float(del_idx.loc["Delayed", "avg_review_score"])
score_gap = (score_on_time - score_delayed) if (score_on_time and score_delayed) else None

# Payments
cc_share = None
if len(df_payments) > 0:
    total_pay_val = df_payments["total_payment_value"].sum()
    cc_pay_val    = df_payments.loc[
        df_payments["payment_type"] == "credit_card", "total_payment_value"
    ]
    cc_share = float(cc_pay_val.values[0]) / total_pay_val * 100 if len(cc_pay_val) > 0 else None

cancel_weighted = (
    df_cancels["canceled_orders"].sum()
    / df_cancels["total_orders"].sum() * 100
)

print("Executive KPI Table")
print("=" * 60)
kpi_summary = {
    "Total Platform GMV (BRL)":       f"{total_gmv_exec:,.0f}",
    "Total Delivered Orders":         f"{total_orders_exec:,.0f}",
    "Average Order Value (BRL)":      f"{avg_aov:,.2f}",
    "Avg Month-1 Retention (%)":      f"{avg_m1_ret:.1f}" if avg_m1_ret else "N/A",
    "Peak Avg Cumulative LTV (BRL)":  f"{peak_ltv:,.2f}",
    "Overall Delay Rate (%)":         f"{delayed_rate_exec:.2f}",
    "Avg ETA Overshoot (days)":       f"{avg_overshot_days:+.1f}",
    "Score Gap (On-Time vs Delayed)": f"{score_gap:.2f}" if score_gap else "N/A",
    "Credit Card Revenue Share (%)":  f"{cc_share:.1f}" if cc_share else "N/A",
    "Weighted Cancel Rate (%)":       f"{cancel_weighted:.2f}",
}
for k, v in kpi_summary.items():
    print(f"  {k:45s}  {v}")


## Analytical Methodology

**Method:** Cross-domain KPI aggregation and risk matrix synthesis.

The executive dashboard does not introduce new analytical methods — it applies the findings from notebooks 01–09 in a unified summary view. The design follows a top-down logic:

1. **Overall performance snapshot** (KPI scorecard, panel A) — establishes baseline before showing drivers.
2. **Most critical time-series signal** (revenue trend, panel B) — the single most-watched operational metric.
3. **Most critical operational risk** (SLA donut, panel C) — the highest-impact operational improvement lever.
4. **Most critical satisfaction signal** (review score distribution, panel D) — reflects the customer experience outcome of all operational decisions.
5. **Most critical financial risk** (payment share, panel E) — revenue concentration and cancellation risk.
6. **Prioritised risk summary table** (panel F) — synthesises all themes into a single decision-support matrix with explicit priority ratings.

**Priority ratings** (High / Medium / Low) are derived from the cross-domain findings in notebooks 01–09. They reflect the combination of financial impact, operational controllability, and finding strength — not subjective judgment.


In [ ]:
# =============================================================================
# Dashboard 10 — Executive Synthesis Dashboard
# =============================================================================
fig = plt.figure(figsize=(18, 16))
fig.suptitle(
    "Olist E-Commerce — Executive Analytical Dashboard",
    fontsize=17, fontweight="bold", y=0.99,
)

# ---------------------------------------------------------------------------
# Panel A (top, wide): Executive KPI scorecard (text table)
# ---------------------------------------------------------------------------
ax_kpi = fig.add_subplot(4, 2, (1, 2))
ax_kpi.axis("off")

kpi_rows = list(kpi_summary.items())
n_cols = 5
n_rows_kpi = 2

ax_kpi.text(0.0, 1.02, "A  |  Platform Performance Snapshot", transform=ax_kpi.transAxes,
            fontsize=12, fontweight="bold")

cell_w = 1.0 / n_cols
cell_h = 0.45

for col_i, (label, val) in enumerate(kpi_rows[:n_cols]):
    x = col_i * cell_w + cell_w / 2
    ax_kpi.text(x, 0.92, label, transform=ax_kpi.transAxes,
                ha="center", fontsize=8.5, color="#555")
    ax_kpi.text(x, 0.68, val, transform=ax_kpi.transAxes,
                ha="center", fontsize=14, fontweight="bold", color="#212121")

for col_i, (label, val) in enumerate(kpi_rows[n_cols:n_cols * 2]):
    x = col_i * cell_w + cell_w / 2
    is_risk = "Delay" in label or "Cancel" in label or "Gap" in label
    ax_kpi.text(x, 0.42, label, transform=ax_kpi.transAxes,
                ha="center", fontsize=8.5, color="#555")
    ax_kpi.text(x, 0.18, val, transform=ax_kpi.transAxes,
                ha="center", fontsize=14, fontweight="bold",
                color="#F44336" if is_risk else "#212121")

# ---------------------------------------------------------------------------
# Panel B (second row left): Revenue trend — high-level sparkline
# ---------------------------------------------------------------------------
ax_rev = fig.add_subplot(4, 2, 3)

ax_rev.fill_between(
    df_monthly["revenue_month"],
    df_monthly["total_revenue"],
    alpha=0.2, color="#2196F3",
)
ax_rev.plot(
    df_monthly["revenue_month"],
    df_monthly["total_revenue"],
    color="#2196F3", linewidth=2,
)
ax_rev.set_title("B  |  Monthly Revenue Trend", loc="left", pad=8)
ax_rev.set_ylabel("Revenue (BRL)")
ax_rev.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax_rev.set_xlabel("")
ax_rev.tick_params(axis="x", rotation=30)
ax_rev.grid(True, axis="y", linestyle="--", alpha=0.5)

# ---------------------------------------------------------------------------
# Panel C (second row right): SLA donut
# ---------------------------------------------------------------------------
ax_sla = fig.add_subplot(4, 2, 4)

total_del    = int(df_sla_exec["total_deliveries"].iloc[0])
total_delay  = int(df_sla_exec["total_delayed_orders"].iloc[0])

wedges, _, autotexts = ax_sla.pie(
    [total_del - total_delay, total_delay],
    labels=["On-Time", "Delayed"],
    autopct="%1.1f%%",
    colors=["#4CAF50", "#F44336"],
    startangle=90,
    wedgeprops=dict(width=0.52),
    textprops=dict(fontsize=10),
)
for at in autotexts:
    at.set_fontweight("bold")
centre_text = f"{delayed_rate_exec:.1f}%\nDelayed"
ax_sla.text(0, 0, centre_text, ha="center", va="center",
            fontsize=12, fontweight="bold", color="#F44336")
ax_sla.set_title("C  |  Delivery SLA Status", loc="left", pad=8)

# ---------------------------------------------------------------------------
# Panel D (third row left): Review score distribution (compact)
# ---------------------------------------------------------------------------
ax_rev_score = fig.add_subplot(4, 2, 5)

score_colors = {1: "#F44336", 2: "#FF7043", 3: "#FF9800", 4: "#8BC34A", 5: "#4CAF50"}
bar_cols = [score_colors.get(int(s), "#9E9E9E") for s in df_scores["review_score"]]
ax_rev_score.bar(
    df_scores["review_score"].astype(str),
    df_scores["pct_of_total"],
    color=bar_cols, width=0.6,
)
ax_rev_score.set_title("D  |  Review Score Distribution (%)", loc="left", pad=8)
ax_rev_score.set_xlabel("Score")
ax_rev_score.set_ylabel("% of Reviews")
ax_rev_score.grid(True, axis="y", linestyle="--", alpha=0.5)

# ---------------------------------------------------------------------------
# Panel E (third row right): Payment method revenue share
# ---------------------------------------------------------------------------
ax_pay = fig.add_subplot(4, 2, 6)

p_colors = {"credit_card": "#2196F3", "boleto": "#FF9800",
            "voucher": "#9C27B0", "debit_card": "#4CAF50"}
pie_cols = [p_colors.get(p, "#607D8B") for p in df_payments["payment_type"]]
ax_pay.pie(
    df_payments["total_payment_value"],
    labels=df_payments["payment_type"].str.replace("_", " ").str.title(),
    autopct="%1.1f%%",
    colors=pie_cols,
    startangle=90,
    textprops=dict(fontsize=9),
)
ax_pay.set_title("E  |  Revenue Share by Payment Method", loc="left", pad=8)

# ---------------------------------------------------------------------------
# Panel F (bottom, wide): Risk matrix — 5 themes × 2 dimensions
# ---------------------------------------------------------------------------
ax_risk = fig.add_subplot(4, 2, (7, 8))
ax_risk.axis("off")

risk_table = [
    # Theme, Lead KPI, Signal, Priority (1=High, 3=Low)
    ("Revenue & AOV",     f"AOV: {avg_aov:,.0f} BRL",       "Flat AOV — volume-driven growth only",              2),
    ("Retention & LTV",   f"M1 Ret: {avg_m1_ret:.1f}%" if avg_m1_ret else "M1 Ret: N/A",
                          "Below-avg retention for majority of cohorts",       1),
    ("Delivery SLA",      f"Delay: {delayed_rate_exec:.1f}%", "Systematic ETA underestimation across network",      1),
    ("Review Scores",     f"Score gap: {score_gap:.2f}" if score_gap else "Gap: N/A",
                          "Delayed orders drive 1-star reviews — recoverable", 1),
    ("Payment Risk",      f"CC share: {cc_share:.1f}%" if cc_share else "CC: N/A",
                          "Single-rail revenue concentration risk",             2),
]

ax_risk.text(0.5, 1.02, "F  |  Cross-Domain Risk Summary",
             transform=ax_risk.transAxes, ha="center",
             fontsize=12, fontweight="bold")

col_headers = ["Analytical Theme", "Lead KPI", "Key Risk Signal", "Priority"]
col_xs      = [0.0, 0.22, 0.44, 0.90]
header_y    = 0.90
ax_risk.text(col_xs[0], header_y, col_headers[0], transform=ax_risk.transAxes,
             fontsize=9, fontweight="bold", color="#333")
ax_risk.text(col_xs[1], header_y, col_headers[1], transform=ax_risk.transAxes,
             fontsize=9, fontweight="bold", color="#333")
ax_risk.text(col_xs[2], header_y, col_headers[2], transform=ax_risk.transAxes,
             fontsize=9, fontweight="bold", color="#333")
ax_risk.text(col_xs[3], header_y, col_headers[3], transform=ax_risk.transAxes,
             fontsize=9, fontweight="bold", color="#333", ha="center")

priority_color = {1: "#F44336", 2: "#FF9800", 3: "#4CAF50"}
priority_label = {1: "HIGH", 2: "MEDIUM", 3: "LOW"}

for row_i, (theme, kpi_val, signal, priority) in enumerate(risk_table):
    y_row = 0.76 - row_i * 0.14
    ax_risk.text(col_xs[0], y_row, theme,     transform=ax_risk.transAxes, fontsize=9)
    ax_risk.text(col_xs[1], y_row, kpi_val,   transform=ax_risk.transAxes, fontsize=9, fontweight="bold")
    ax_risk.text(col_xs[2], y_row, signal,    transform=ax_risk.transAxes, fontsize=8.5, color="#444")
    ax_risk.text(col_xs[3], y_row, priority_label[priority],
                 transform=ax_risk.transAxes, fontsize=9, fontweight="bold",
                 color=priority_color[priority], ha="center")

plt.tight_layout(rect=[0, 0, 1, 0.97])
save_figure(fig, "10_executive_dashboard.png")
plt.show()


# Executive Analytical Dashboard & Synthesis — Conclusions

---

## Key Findings
- Revenue growth heavily relies on sustained acquisition volume, driven by flat structural AOV and a narrow 2-3 month monetization plateau.
- Systematic delivery ETA undercuts occur nationwide, culminating directly in lower average review scores for delayed orders.
- The credit card payment rail independently supports the vast majority of platform GMV, concentrating operational infrastructure risk.
- High-density logistics and high-volume revenue share identical geographic footprints, compounding regional point-of-failure vulnerabilities.
- Unpredictable short-term revenue noise masks highly measurable, regular long-term seasonality peaks and long-tail outliers.

## Business Implications
- The causal relationship between delivery SLA failures and 1-star reviews provides a single operational lever to improve brand perception metrics.
- Over-reliance on volume growth without complementary improvements to AOV or retention mathematically establishes an impending growth ceiling.
- Applying uniform, unmodified logistics strategies across varying state profiles and outlier tiers inefficiently misallocates problem-solving capital.

## Actionable Recommendations
- Recalibrate public-facing delivery ETAs instantly using state-specific historical median transit times to neutralize broken-promise friction.
- Monitor the credit card payment gateway success rate continuously, backed by automated redundancy switching to protect dominant revenue lines.
- Implement an explicitly weighted state-level triage model for carrier escalation focusing entirely on the high-GMV, above-median-delay quadrants.
- Standardize the usage of median central tendencies and 3-month rolling averages across all executive reporting frameworks.
